In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import time
import torch

c:\Users\Admin\Desktop\ip\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODELO_PATH = r"C:\Users\Admin\Desktop\models\Teste\Qwen3-4B-Base\BASE"

modelo = AutoModelForCausalLM.from_pretrained (MODELO_PATH, device_map = "cpu")

tokenizer = AutoTokenizer.from_pretrained (MODELO_PATH)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 1079.85it/s]


<h1> Time to First Token </h1>

processa prompt (prefill) 
constrói KV cache
gera primeiro token
retorna output

In [31]:
TtFT = []

prompt = "O que sabes da empresa Infraestruturas de Portugal ?"

input = tokenizer (prompt, return_tensors = "pt")
#print (len(input["input_ids"])) #TESTE -> 14

for _ in range (35):
    
    t0 = time.perf_counter()
    tokens = modelo.generate (**input, max_new_tokens = 1)

    t1 = time.perf_counter()

    delta = t1 - t0
    TtFT.append(delta)


[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_toke

In [32]:
print (TtFT)
print (len(TtFT))

[1.7920844000000216, 1.5263016999999763, 1.5218257999999878, 1.5380958999999166, 1.5348765999999614, 1.536232899999959, 1.5545011000001523, 1.5881632000000536, 1.7942344999999023, 1.7695811000000958, 1.7308573000000251, 1.7200886000000537, 1.7258646000000226, 1.7335607999998501, 1.734983100000136, 1.6029393000001164, 1.564358299999867, 1.5594427000000906, 1.5527870000000803, 1.5733338000000003, 1.552014900000131, 1.5499601999999868, 1.5824895000000652, 1.5702616999999464, 1.5492987000000085, 1.5469837000000553, 1.546567399999958, 1.555822600000056, 1.5821260000000166, 1.5697606999999607, 1.5425242999999682, 1.5750368999999864, 1.5775123000000804, 1.557877499999904, 1.5320438999999624]
35


# Latencia Total


Calculo do tempo total desde tokenizer ate geracao total

In [40]:
Lat = []

prompt = "O que sabes da empresa Infraestruturas de Portugal ?"

for i in range (15):

    t0 = time.perf_counter()
    input = tokenizer (prompt, return_tensors = "pt")

    tokens = modelo.generate (**input, max_new_tokens = 100)

    texto = tokenizer.decode (tokens[0])

    #print (texto)
    t1 = time.perf_counter()

    delta = t1 - t0
    Lat.append(delta)
    

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
[transformers] Setting `pad_toke

In [41]:
print (Lat)

[28.845445200000086, 30.20715549999977, 29.162379200000032, 29.513841799999682, 28.60348789999989, 28.970935499999996, 29.597843399999874, 28.55963470000006, 29.222640899999988, 29.1251771000002, 28.00827990000016, 28.10415779999994, 28.28073039999981, 28.206465299999763, 28.289368900000227]


# Throughput

### Prefill = Prompt Processing

### Decode = Token Processing


In [9]:
import torch.nn.functional as F

throughput = {
    "Prefill": [],
    "Decode": []
}

prompt = "O que sabes da empresa Infraestruturas de Portugal ?"

input = tokenizer (prompt, return_tensors = "pt")
#print (len(input["input_ids"])) #TESTE -> 14

input_ids = input['input_ids']

for i in range (15):
    
    #PREFILL
    t0 = time.perf_counter()
    logits = modelo (input_ids = input_ids)
    t1 = time.perf_counter()

    #DECODE
    generated = input_ids

    t2 = time.perf_counter()
    for _ in range (10):

        outputs = modelo (input_ids = generated)


        logits = outputs.logits [:, -1, :]
        probs = F.softmax (logits, dim = -1)

        next_token = torch.multinomial (probs, num_samples = 1)

        generated = torch.cat ([generated, next_token], dim = 1)

        #print (tokenizer.decode (generated[0]))
    t3 = time.perf_counter()

    throughput["Prefill"].append(t1 - t0)
    throughput["Decode"].append(t3 - t2)


In [10]:
print (throughput)

{'Prefill': [1.880804000000353, 1.9373442999931285, 1.7754378000099678, 1.7660962999943877, 1.9236811999871861, 1.9399644999939483, 1.9835797999985516, 1.9940624000009848, 2.08306979999179, 1.7435592999972869, 1.930250100005651, 1.8739883000089321, 1.7129140000033658, 1.938071900003706, 1.9758846999902744], 'Decode': [26.066857500001788, 25.973421700007748, 22.91365119999682, 25.343393999995897, 26.512460300000384, 25.613794799995958, 25.8587114999973, 26.850021200007177, 23.54651919999742, 22.676711000007344, 23.90458080000826, 22.88175119999505, 25.304488299996592, 26.503566400002455, 24.561820100003388]}
